# Heston exotic options — individual calibrations

Compare floating-strike lookbacks, down-and-out calls, and forward-start calls
for Heston and the two saved individual calibrations:

- Piecewise constant: `single/result.json`.
- Legendre: `single/result.json`.

Each model uses the configuration and coefficients stored in its result file.
Prices are saved in `Heston/Exotic option results/individual_calibrations_1M_current_cpu`.
The six paper-format figures are shown inline and saved under
`Heston/Plots/Exotic_options`. Replotting needs no simulation.

Restart the kernel after code changes before running this notebook.


In [ ]:
from pathlib import Path
import sys
import importlib

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "Heston/exotic_option_study.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project repository.")
for path in (ROOT, ROOT / "Heston"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import exotic_option_study as exotic
importlib.reload(exotic)

RESULTS = ROOT / exotic.DEFAULT_RESULTS
PLOTS = ROOT / "Heston/Plots/Exotic_options"
RUN_PRICING = False  # Set True only to create or resume Monte Carlo checkpoints.
PRICING_DEVICE = "cpu"  # Match the saved checkpoint manifest when resuming.
MC_BUDGET = {"heston": (10_000, 100), "chaos": (10_000, 100)}  # 1M paths per model.
BASE_SEED = 42
print(f"Checkpoints: {RESULTS / 'data'}\nPaper-format figures: {PLOTS}")
for label, path in exotic.RUN_FILES.items():
    print(f"{label}: {path}")


## Price or resume Monte Carlo

Set `RUN_PRICING = True` to compute **1 million paths per model**, in 100 batches
of 10,000. Each batch simulates once to the last expiry and prices all maturities,
strikes, barriers and forward starts from those same paths. Heston uses QE-M in
Torch; the chaos paths and payoff calculations use the selected device.
The progress bar advances after every batch. Completed models are checkpointed.
Choose the device with `PRICING_DEVICE`. Existing study-run checkpoints are
preserved separately. This notebook uses only the individual-calibration results.
Replotting existing results needs no simulation. Pricing rejects checkpoints created
with different code, data, or Monte Carlo settings.

Only the contracts shown in the paper figures are priced:

- **Down-and-out calls:** maturities `0.1, 0.5, 1.0`, barriers `80, 90`,
  strikes `95, 100, 105, 110, 115, 120`.
- **Forward-start calls:** reset times `0.1, 0.2, 0.3`, expiry `1.0`,
  moneyness `0.9, 0.95, 1.0, 1.05, 1.1`.
- **Lookbacks:** maturities `0.1, 0.2, ..., 1.0`.

Heston and both chaos models use a simulation/monitoring step of `0.001`
(1,000 steps to expiry `1.0`). The saved prices use this grid.



In [ ]:
if RUN_PRICING:
    import pickle
    import torch

    device = torch.device(PRICING_DEVICE)
    with (ROOT / "Heston/heston_data.pkl").open("rb") as handle:
        data = pickle.load(handle)
    models, coefficients, RESULTS = exotic.prepare_calibrations(
        ROOT, data, device, MC_BUDGET, seed=BASE_SEED, output=RESULTS,
    )
    exotic.run_pricing(
        data, models, coefficients, RESULTS, device, MC_BUDGET, seed=BASE_SEED,
    )
    print(f"Completed exotic-option grids in {RESULTS / 'data'}")
else:
    print("Using saved Monte Carlo checkpoints; no simulation needed.")

## Save and show the six paper-format figures

Each payoff has a price panel and a Black–Scholes equivalent IV panel. IVs are
recomputed from saved prices using the current formulas. Lookback and barrier
IVs use continuous-monitoring formulas; Monte Carlo prices use the discrete grid.
Undefined or nonunique barrier IVs appear as gaps. All three models are labelled.


In [ ]:
import importlib
import pickle
from wiener_chaos.pricing import options as implied_vol

# Refresh lookback IVs from cached prices without repeating Monte Carlo.
importlib.reload(implied_vol)
importlib.reload(exotic)
with (ROOT / "Heston/heston_data.pkl").open("rb") as handle:
    data = pickle.load(handle)

if (RESULTS / "data").is_dir():
    figures = exotic.plot_results(RESULTS, PLOTS, data=data)
    for path in figures:
        print(path.relative_to(ROOT))
else:
    print("No exotic prices saved for the individual calibrations yet. "
          "Set RUN_PRICING = True and rerun the pricing cell first.")
